In [1]:
"""
STEP 0 (runs BEFORE tier 1/2/3): abbreviation expansion.

Finds "full phrase (ABBR)" style definitions within each paper's own
sentences, then expands bare "ABBR" occurrences in that paper's entity
names back to the full phrase, e.g. "corn gluten meal (CGM)" defined
once, then "CGM" anywhere else in that same paper's entities becomes
"corn gluten meal".

Pure regex, no LLM or embedding calls, no OPENAI_API_KEY needed. Runs
first because every downstream step (family tagging, code-based merge,
LLM adjudication) works off entity text, and abbreviation differences
add noise to similarity checks and can hide real distinctions (e.g. a
"gel" state modifier could be invisible if it's folded into an
abbreviation the pipeline doesn't recognize).

Method:
  1. Scan each paper's sentences for "<phrase> (<ABBR>)" patterns.
  2. Validate: do the abbreviation's letters match the initials of the
     phrase's words, in order (skipping minor words like "of", "the")?
     This filters out coincidental parentheticals that aren't real
     abbreviation definitions.
  3. Build a lookup PER PAPER (doi -> {abbr: full_phrase}), not global,
     since the same abbreviation could theoretically mean something
     different in a different paper. Confirmed real in this corpus:
     "PPI" = "pea protein isolate" in one paper, "perilla protein
     isolate" in another. A global entry for PPI would silently corrupt
     whichever paper doesn't match.
  4. Expand bare abbreviation mentions in that paper's entity names
     using the per-paper lookup first.
  5. SECOND PASS: for anything still unresolved after step 4, apply a
     small STATIC global dictionary, restricted to standard chemistry
     reagents and analytical methods that mean the same thing in every
     paper (KOH, DPPH, FTIR, etc.), never material/sample names, which
     is exactly the category that's unsafe to treat globally (see PPI
     above). This catches cases like "CGM albumin protein" where the
     defining sentence was never captured for this paper at all, so
     there was nothing for the per-paper pass to find.
  6. Whatever is STILL unresolved after both passes gets reported
     separately, this is the genuinely paper-specific residual (~15-20
     papers), worth a full-text pull or acceptance as unexpanded,
     rather than guessed at.

Output is a REVIEW file. Nothing is auto-applied to your main triples
file, check the detected definitions before trusting them.

Designed for Jupyter/Colab execution. No __main__ guard.
"""

import re
import pandas as pd

# ---------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------
TRIPLES_PATH = r"C:\Users\olagunju\OneDrive\KSU PROJECT\SHOLA_KSU_PUBLISHED_PAPERS\Data-Minning\NER-PROJECT\PART-3-PREPROCESSING-TRIPLICATE\phase2_terra_full_triples_categorized.xlsx" # swap to full corpus file when ready
SOURCE_COL = "source"
TARGET_COL = "target"
DOI_COL = "doi"
SENTENCE_COL = "corresponding_sentence"

MINOR_WORDS = {"a", "an", "the", "of", "in", "on", "for", "to", "with", "and", "or", "at", "from"}
MAX_PHRASE_WORDS = 6  # how many words back to look for a matching definition

# Static, field-wide dictionary: standard chemistry reagents and
# analytical methods only, NEVER material/sample names. These mean the
# same thing regardless of which paper uses them, unlike PPI, NCP, SPI,
# etc., which are locally-scoped and MUST stay resolved per-paper.
# Expand this list as you find more genuinely universal cases, keep
# material/sample abbreviations OUT of it.
GLOBAL_ABBREVIATIONS = {
    "KOH": "potassium hydroxide",
    "NAOH": "sodium hydroxide",
    "HCL": "hydrochloric acid",
    "MDA": "malondialdehyde",
    "AAPH": "2,2'-azobis(2-amidinopropane) dihydrochloride",
    "DPPH": "2,2-diphenyl-1-picrylhydrazyl",
    "ABTS": "2,2'-azino-bis(3-ethylbenzothiazoline-6-sulfonic acid)",
    "SDS": "sodium dodecyl sulfate",
    "FTIR": "fourier-transform infrared spectroscopy",
    "SEM": "scanning electron microscopy",
    "XRD": "x-ray diffraction",
    "NMR": "nuclear magnetic resonance",
    "DSC": "differential scanning calorimetry",
    "ITC": "isothermal titration calorimetry",
    "CD": "circular dichroism",
    "ANN": "artificial neural network",
    "GA": "genetic algorithm",
    "PBS": "phosphate-buffered saline",
    "EDTA": "ethylenediaminetetraacetic acid",
    "BSA": "bovine serum albumin",
}

OUTPUT_XLSX = "abbreviation_expansion_review.xlsx"

# ---------------------------------------------------------------
# STEP 1: FIND CANDIDATE "<phrase> (<ABBR>)" PATTERNS IN SENTENCES
# ---------------------------------------------------------------
# a phrase of 1-6 words, followed by "(ABBR)" where ABBR is 2-8 chars,
# starts with a capital letter, letters/digits only
_DEFINITION_PATTERN = re.compile(
    r"((?:[A-Za-z][A-Za-z\-]*\s+){0,%d}[A-Za-z][A-Za-z\-]*)\s*\(([A-Z][A-Za-z0-9]{1,7})\)"
    % (MAX_PHRASE_WORDS - 1)
)


def initials_match(phrase, abbr):
    """
    Does abbr's letters match the initials of phrase's words, in order,
    allowing minor words (of, the, and, etc.) to be skipped? Returns
    True/False. This is what filters real abbreviation definitions
    ("corn gluten meal (CGM)") from coincidental parentheticals
    ("protein content (95%)", "sample size (N)").
    """
    words = [w for w in re.split(r"[\s\-]+", phrase.strip()) if w]
    content_words = [w for w in words if w.lower() not in MINOR_WORDS]
    if not content_words:
        return False
    initials = "".join(w[0] for w in content_words).upper()
    abbr_letters = "".join(c for c in abbr.upper() if c.isalpha())
    if not abbr_letters:
        return False
    # abbr's letters should appear, in order, within the content-word
    # initials (allows the phrase to have a couple extra leading words
    # beyond what the abbreviation actually covers)
    return abbr_letters in initials or initials.endswith(abbr_letters) or initials == abbr_letters


def find_shortest_matching_phrase(candidate_words, abbr):
    """
    Try progressively shorter word windows (from the end, closest to
    the parenthesis, backward) and return the shortest one whose
    initials match abbr. This avoids grabbing extra unrelated words
    from earlier in the sentence.
    """
    for n in range(1, len(candidate_words) + 1):
        window = candidate_words[-n:]
        phrase = " ".join(window)
        if initials_match(phrase, abbr):
            return phrase
    return None


# ---------------------------------------------------------------
# LOAD DATA
# ---------------------------------------------------------------
def load_triples(path):
    if path.endswith(".parquet"):
        return pd.read_parquet(path)
    elif path.endswith(".xlsx"):
        return pd.read_excel(path)
    elif path.endswith(".csv"):
        return pd.read_csv(path)
    else:
        raise ValueError(f"Unsupported file type: {path}")


df = load_triples(TRIPLES_PATH)

# one row per unique (doi, sentence)
sentences_by_doi = (
    df[[DOI_COL, SENTENCE_COL]]
    .dropna(subset=[SENTENCE_COL])
    .drop_duplicates()
    .groupby(DOI_COL)[SENTENCE_COL]
    .apply(list)
    .to_dict()
)
print(f"Papers with sentence data: {len(sentences_by_doi)}")

# ---------------------------------------------------------------
# EXTRACT DEFINITIONS PER PAPER
# ---------------------------------------------------------------
definitions = []  # doi, abbr, full_phrase, source_sentence

for doi, sentences in sentences_by_doi.items():
    for sentence in sentences:
        for match in _DEFINITION_PATTERN.finditer(sentence):
            raw_phrase, abbr = match.group(1), match.group(2)
            # skip abbreviations containing digits (T2, H1, C13, etc.):
            # these are almost always technical notation (NMR relaxation
            # times, isotope labels), not word-initial abbreviations, and
            # stripping the digit for the initials check causes false
            # matches, e.g. "T2" matching "time" because "T" == "T" once
            # the "2" is thrown away
            if any(c.isdigit() for c in abbr):
                continue
            candidate_words = [w for w in re.split(r"\s+", raw_phrase.strip()) if w]
            phrase = find_shortest_matching_phrase(candidate_words, abbr)
            if phrase:
                definitions.append({
                    "doi": doi,
                    "abbreviation": abbr,
                    "full_phrase": phrase.lower(),
                    "source_sentence": sentence,
                })

definitions_df = pd.DataFrame(definitions).drop_duplicates(subset=["doi", "abbreviation", "full_phrase"])
print(f"Candidate definitions found: {len(definitions_df)}")

# if the same (doi, abbreviation) matched more than one phrase across
# different sentences, keep the most frequent one and flag the rest
# for review rather than silently picking one
definitions_df["dupe_count"] = definitions_df.groupby(["doi", "abbreviation"])["full_phrase"].transform("count")
ambiguous_df = definitions_df[definitions_df["dupe_count"] > 1].sort_values(["doi", "abbreviation"])
clean_df = (
    definitions_df.sort_values("dupe_count", ascending=False)
    .drop_duplicates(subset=["doi", "abbreviation"], keep="first")
)
print(f"Clean (doi, abbreviation) pairs: {len(clean_df)}")
print(f"Ambiguous, same abbreviation matched >1 phrase in same paper: {ambiguous_df['doi'].nunique()} papers affected")

# per-doi lookup dict for the expansion step
lookup = {}
for _, row in clean_df.iterrows():
    lookup.setdefault(row["doi"], {})[row["abbreviation"]] = row["full_phrase"]

# ---------------------------------------------------------------
# EXPAND ABBREVIATIONS IN ENTITY NAMES
# PASS 1: per-paper lookup (highest confidence, paper-scoped)
# PASS 2: static global dictionary, ONLY for tokens pass 1 didn't touch
# ---------------------------------------------------------------
def expand_entity(entity, doi):
    """Returns (expanded_text, resolved_by_per_paper, resolved_by_global)."""
    if pd.isna(entity):
        return entity, False, False
    text = str(entity)
    changed_per_paper = False
    changed_global = False

    # pass 1: per-paper lookup
    for abbr, full_phrase in lookup.get(doi, {}).items():
        pattern = re.compile(rf"\b{re.escape(abbr)}\b")
        if pattern.search(text):
            text = pattern.sub(full_phrase, text)
            changed_per_paper = True

    # pass 2: static global dictionary, only for tokens still present
    # after pass 1 (avoids re-processing something already resolved)
    for abbr, full_phrase in GLOBAL_ABBREVIATIONS.items():
        pattern = re.compile(rf"\b{re.escape(abbr)}\b", re.IGNORECASE)
        if pattern.search(text):
            text = pattern.sub(full_phrase, text)
            changed_global = True

    return text, changed_per_paper, changed_global


expanded_rows = []
for _, row in df.iterrows():
    src_expanded, src_pp, src_gl = expand_entity(row[SOURCE_COL], row[DOI_COL])
    tgt_expanded, tgt_pp, tgt_gl = expand_entity(row[TARGET_COL], row[DOI_COL])
    if src_pp or src_gl or tgt_pp or tgt_gl:
        expanded_rows.append({
            "doi": row[DOI_COL],
            "original_source": row[SOURCE_COL],
            "expanded_source": src_expanded,
            "original_target": row[TARGET_COL],
            "expanded_target": tgt_expanded,
            "resolved_by": "; ".join(sorted({
                *(["per_paper"] if (src_pp or tgt_pp) else []),
                *(["global_dict"] if (src_gl or tgt_gl) else []),
            })),
        })

expanded_df = pd.DataFrame(expanded_rows)
print(f"\nRows with at least one abbreviation expanded: {len(expanded_df)}")
print(f"  Resolved via per-paper definition: {(expanded_df['resolved_by'].str.contains('per_paper')).sum() if len(expanded_df) else 0}")
print(f"  Resolved via static global dictionary: {(expanded_df['resolved_by'].str.contains('global_dict')).sum() if len(expanded_df) else 0}")

# ---------------------------------------------------------------
# RESIDUAL: entities with an all-caps token that NEITHER pass resolved.
# This is the genuinely paper-specific "category 1" set, worth a
# full-text pull or acceptance as unexpanded, not a guess.
# ---------------------------------------------------------------
ABBR_TOKEN = re.compile(r"\b[A-Z]{2,8}\b")

entities_long = pd.concat([
    df[[SOURCE_COL, DOI_COL]].rename(columns={SOURCE_COL: "entity"}),
    df[[TARGET_COL, DOI_COL]].rename(columns={TARGET_COL: "entity"}),
]).dropna().drop_duplicates()

residual_rows = []
for _, row in entities_long.iterrows():
    entity, doi = str(row["entity"]), row[DOI_COL]
    tokens = ABBR_TOKEN.findall(entity)
    known_per_paper = lookup.get(doi, {})
    for tok in tokens:
        if tok not in known_per_paper and tok.upper() not in GLOBAL_ABBREVIATIONS:
            residual_rows.append({"doi": doi, "entity": entity, "undefined_token": tok})

residual_df = pd.DataFrame(residual_rows).drop_duplicates()
print(f"\nStill unresolved after both passes (category 1, paper-specific): "
      f"{residual_df['entity'].nunique() if len(residual_df) else 0} entities across "
      f"{residual_df['doi'].nunique() if len(residual_df) else 0} papers")

# ---------------------------------------------------------------
# SAVE FOR REVIEW
# ---------------------------------------------------------------
with pd.ExcelWriter(OUTPUT_XLSX) as writer:
    clean_df.drop(columns="dupe_count").to_excel(writer, sheet_name="detected_definitions", index=False)
    ambiguous_df.to_excel(writer, sheet_name="ambiguous_definitions", index=False)
    expanded_df.to_excel(writer, sheet_name="expanded_entities", index=False)
    residual_df.to_excel(writer, sheet_name="still_unresolved", index=False)

print(f"\nSaved to {OUTPUT_XLSX}")
print("Review detected_definitions before trusting them, spot-check a sample.")
print("ambiguous_definitions: same abbreviation, multiple possible phrases in one paper, needs a manual look.")
print("expanded_entities: before/after view of every entity that got expanded, resolved_by shows which pass did it.")
print("still_unresolved: category 1, genuinely paper-specific, decide: pull full text or accept as unexpanded.")
print("\nOnce approved, run this BEFORE tier1_family_tagging.py / tier2 / tier3,")
print("using expanded_source/expanded_target in place of source/target.")

Papers with sentence data: 192
Candidate definitions found: 186
Clean (doi, abbreviation) pairs: 181
Ambiguous, same abbreviation matched >1 phrase in same paper: 3 papers affected

Rows with at least one abbreviation expanded: 168
  Resolved via per-paper definition: 47
  Resolved via static global dictionary: 121

Still unresolved after both passes (category 1, paper-specific): 268 entities across 78 papers

Saved to abbreviation_expansion_review.xlsx
Review detected_definitions before trusting them, spot-check a sample.
ambiguous_definitions: same abbreviation, multiple possible phrases in one paper, needs a manual look.
expanded_entities: before/after view of every entity that got expanded, resolved_by shows which pass did it.
still_unresolved: category 1, genuinely paper-specific, decide: pull full text or accept as unexpanded.

Once approved, run this BEFORE tier1_family_tagging.py / tier2 / tier3,
using expanded_source/expanded_target in place of source/target.
